# ⚛️ NSL-KDD QSVM — Quantum Circuit Architecture

**Notebook:** `02_Quantum_Circuit_Architecture.ipynb`  
**Project:** Quantum SVM for Network Intrusion Detection (NIDS)  
**Tool:** Qiskit 2.x · ZZFeatureMap · 4 Qubits · reps=3 · entanglement='full'

---

## Overview

This notebook visualises the **ZZFeatureMap quantum feature encoding circuit** used as the kernel function of our newly trained QSVM.  The circuit uses:

```
feature_dimension = 4   → 4 PCA components from Phase 1
reps              = 3   → three repetitions of the H + Phase/ZZ layer
entanglement      = 'full'  → all qubit pairs are entangled (6 ZZ pairs per rep)
```

The same circuit is used in Phases 2–4 of the project:

```
src/data_preprocessing.py   → MinMaxScaler scales features to [0, π]
src/train_qsvm.py           → ZZFeatureMap encodes features as rotation angles
src/evaluate_models.py      → Same circuit used for K_test computation
src/app.py                  → Same circuit used for live inference
```

We visualise the circuit at the **gate-level (decomposed)** abstraction:

| Parameter | Value | Description |
|-----------|-------|-------------|
| `feature_dimension` | 4 | Matches 4 PCA components |
| `reps` | 3 | Three repetitions of H + Phase + ZZ |
| `entanglement` | `'full'` | All qubit pairs connected (6 ZZ pairs per rep) |
| Circuit depth (decomposed) | ~45 | `circuit.decompose().depth()` |

## 0 · Environment Setup

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from qiskit.circuit.library import zz_feature_map

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────
REPORTS = Path("..") / "reports"
os.makedirs(REPORTS, exist_ok=True)

FIGURE_DPI = 300

print("Qiskit and dependencies loaded successfully.")
print(f"Reports directory: {REPORTS.resolve()}")
print(f"Figure DPI: {FIGURE_DPI}")

## 1 · ZZFeatureMap: Mathematical Foundation

### What is the ZZFeatureMap?

The **ZZFeatureMap** (Havlicek et al., *Nature* 2019) is a parameterised quantum circuit that maps a classical feature vector **x** ∈ ℝ⁴ to a quantum state |φ(**x**)⟩ in the 2⁴ = **16-dimensional** Hilbert space ℂ¹⁶.

### The encoding circuit U(**x**)

One "rep" of the circuit applies three sub-layers in sequence:

**Layer 1 — Hadamard superposition**

$$H^{\otimes 4}|0000\rangle = \frac{1}{\sqrt{16}} \sum_{z \in \{0,1\}^4} |z\rangle$$

**Layer 2 — Single-qubit phase encoding**

$$U_1(\mathbf{x}) = \bigotimes_{i=0}^{3} P(2x_i) = \bigotimes_{i=0}^{3} R_Z(2x_i)$$

**Layer 3 — Two-qubit ZZ entangling encoding (full connectivity)**

For each entangled pair $(i,j)$ in `{(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)}`:

$$U_2(x_i, x_j) = \exp\!\left(i\,(\pi - x_i)(\pi - x_j)\, Z_i \otimes Z_j\right)$$

Implemented as: `CX(i→j)` · `Rz(2(π−xᵢ)(π−xⱼ))` on qubit j · `CX(i→j)`.

**Full circuit (reps=3, full entanglement):**

$$U(\mathbf{x}) = \left[H^{\otimes 4} \cdot U_1(\mathbf{x}) \cdot U_2^{\text{full}}(\mathbf{x})\right]^3$$

### Gate count analysis for reps=3, full entanglement

| Gate | Per rep | Total (×3 reps) |
|------|---------|-----------------|
| **H** (Hadamard) | 4 | 12 |
| **P/Rz** (single-qubit) | 4 | 12 |
| **ZZ CX pairs** | 6 pairs × 2 = 12 | 36 |
| **ZZ Rz rotations** | 6 | 18 |

### The quantum kernel

$$K(\mathbf{x}, \mathbf{z}) = \left|\langle\phi(\mathbf{x})|\phi(\mathbf{z})\rangle\right|^2 = \left|\langle 0000 | U^\dagger(\mathbf{z})\, U(\mathbf{x}) | 0000\rangle\right|^2$$

## 2 · Circuit Initialization

In [ ]:
# ── Instantiate ZZFeatureMap: reps=3, entanglement='full' ──────────────────
# Parameters match the newly trained QSVM architecture:
#   feature_dimension=4  → 4 PCA components from Phase 1
#   reps=3               → three repetitions of H + Phase + ZZ layer
#   entanglement='full'  → all qubit pairs entangled: (0,1),(0,2),(0,3),(1,2),(1,3),(2,3)
#
# In Qiskit 2.x the function-based API zz_feature_map() returns a plain
# QuantumCircuit that is mathematically identical to the class-based API.
circuit = zz_feature_map(
    feature_dimension=4,
    reps=3,
    entanglement="full",
)

print("=" * 60)
print("  ZZFeatureMap Circuit Summary")
print("=" * 60)
print(f"  API              : zz_feature_map() [Qiskit 2.x function]")
print(f"  Qubits           : {circuit.num_qubits}")
print(f"  Parameters       : {circuit.num_parameters}  ({list(circuit.parameters)})")
print(f"  High-level depth : {circuit.depth()}")
print(f"  Gate counts      : {dict(circuit.count_ops())}")
print()

decomposed = circuit.decompose()
print(f"  Decomposed depth : {decomposed.depth()}")
print(f"  Decomposed gates : {dict(decomposed.count_ops())}")
print()
print("  Gate legend:")
print("    H  = Hadamard (superposition layer)")
print("    P  = Phase gate = Rz (single-qubit feature encoding)")
print("    CX = CNOT (ZZ entangling interaction)")
print("=" * 60)

## 3 · Decomposed Circuit Diagram

We draw the **decomposed** version of the ZZFeatureMap using `circuit.decompose().draw()`.  
This expands the high-level gates one level to expose the underlying `H`, `P`, and `CX` gates.

- **`fold=50`** wraps the circuit every 50 columns for readability.
- The decomposed depth is approximately **45** for `reps=3, entanglement='full'`.

> **Why `fig.savefig()` and NOT `plt.savefig()`?**  
> `circuit.draw(output='mpl')` returns a **Matplotlib Figure object**.  
> Calling `fig.savefig()` **explicitly targets the circuit figure**, guaranteeing  
> a non-blank saved image regardless of Jupyter's display backend.

In [ ]:
# ── Draw the decomposed circuit ────────────────────────────────────────────
# circuit.decompose() unfolds the high-level gates one level, exposing
# H, P (Phase/Rz), and CX gates explicitly.
# fold=50 wraps the diagram every 50 columns for readability.
#
# CRITICAL: capture the Figure object returned by draw(); use fig.savefig().
# DO NOT call plt.savefig() — it targets the matplotlib "current figure"
# which may differ in a Jupyter session, yielding blank output.
fig = circuit.decompose().draw(output="mpl", fold=50)

fig.suptitle(
    "ZZFeatureMap — Decomposed Circuit\n"
    "feature_dimension=4 | reps=3 | entanglement='full' | fold=50",
    fontsize=11, fontweight="bold", y=1.02,
)

# ── Save to reports/ ───────────────────────────────────────────────────────
out_path = REPORTS / "quantum_circuit_reps3_full.png"
fig.savefig(out_path, dpi=FIGURE_DPI, bbox_inches="tight")
print(f"Figure saved → {out_path}  ({out_path.stat().st_size / 1024:.1f} KB)")
print(f"Decomposed circuit depth: {circuit.decompose().depth()}")

display(fig)
plt.close(fig)